# External Validation

MIMIC-IV로 학습된 모델(`models/optuna/`)을 **외부 데이터셋**에서 검증하는 노트북.

### 사용법
1. `EXTERNAL_DATA_PATH`에 외부 데이터 CSV 경로를 지정
2. `TARGET_COL`이 외부 데이터에 존재하는지 확인 (없으면 예측만 수행)
3. 컬럼명이 MIMIC-IV 학습 데이터와 동일해야 함 (아래 `FEATURE_COLS` 참조)

### 저장된 모델
| Model | File | Scaler |
|-------|------|--------|
| LR | `lr_model.joblib` | `lr_scaler.joblib` |
| RF | `rf_model.joblib` | - |
| XGB | `xgb_model.joblib` | - |
| MLP | `mlp_model.pt` | `mlp_scaler.joblib` |

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import warnings
warnings.filterwarnings('ignore')

import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_curve, auc, precision_recall_curve,
    confusion_matrix, recall_score, matthews_corrcoef,
    classification_report
)

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

print('Imports OK')

In [ ]:
# ============================================================
# Configuration — 여기만 수정하세요
# ============================================================
MODEL_DIR = '../models/optuna'          # 저장된 모델 디렉토리
EXTERNAL_DATA_PATH = '../Data/external_dataset.csv'  # <-- 외부 데이터 경로

TARGET_COL = 'CAM-ICU'                  # 타겟 컬럼 (없으면 예측만 수행)

# 학습 시 사용된 피처 컬럼 (순서 중요)
FEATURE_COLS = [
    'age', 'gender',
    'Heart Rate_mean', 'Mean BP_mean', 'Oxygen Saturation_mean', 'Temperature_mean',
    'Heart Rate_std', 'Mean BP_std', 'Oxygen Saturation_std', 'Temperature_std',
    'Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine', 'Vasopressors', 'Ventilator',
    'Weight', 'RASS',
    'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets',
    'Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Calcium', 'Magnesium', 'Phosphate',
    'BUN', 'Creatinine', 'ALT',
    'Glucose', 'Lactate', 'pH', 'pCO2', 'pO2'
]

# 바이너리 컬럼 (outlier 처리 제외 대상)
BINARY_COLS = ['gender', 'Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine', 'Vasopressors', 'Ventilator']

THRESHOLD = 0.5

print(f'Model dir : {MODEL_DIR}')
print(f'Data path : {EXTERNAL_DATA_PATH}')
print(f'Features  : {len(FEATURE_COLS)}')

## 2. MLP Architecture (학습 시와 동일)

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x)

print('MLP class defined')

## 3. Evaluation Functions

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    """Compute classification metrics."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = auc(fpr, tpr)

    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    auc_prc = average_precision_score(y_true, y_prob)

    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    ppv = tp / max(tp + fp, 1)
    npv = tn / max(tn + fn, 1)
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    mcc = matthews_corrcoef(y_true, y_pred)

    idx_90 = np.argmin(np.abs(tpr - 0.90))
    spec_at_90 = 1 - fpr[idx_90]

    return {
        'auc': auc_val,
        'auc_prc': auc_prc,
        'ppv': ppv,
        'npv': npv,
        'mcc': mcc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'spec@90': spec_at_90,
        'fpr': fpr,
        'tpr': tpr,
        'precision_curve': precision_curve,
        'recall_curve': recall_curve,
    }

print('compute_metrics defined')

## 4. Load External Data

In [ ]:
df_ext = pd.read_csv(EXTERNAL_DATA_PATH)
print(f'External dataset shape: {df_ext.shape}')
print(f'Columns: {list(df_ext.columns)}')

# 타겟 존재 여부 확인
HAS_LABEL = TARGET_COL in df_ext.columns
print(f'\nTarget column "{TARGET_COL}" present: {HAS_LABEL}')

if HAS_LABEL:
    print(f'  Positive: {(df_ext[TARGET_COL] == 1).sum():,} ({df_ext[TARGET_COL].mean()*100:.2f}%)')
    print(f'  Negative: {(df_ext[TARGET_COL] == 0).sum():,} ({(1-df_ext[TARGET_COL].mean())*100:.2f}%)')

In [ ]:
# 피처 컬럼 검증
missing_cols = [c for c in FEATURE_COLS if c not in df_ext.columns]
extra_cols = [c for c in df_ext.columns if c not in FEATURE_COLS + ['stay_id', 'cam_bin', TARGET_COL, 'los']]

if missing_cols:
    print(f'WARNING: 누락된 피처 컬럼 {len(missing_cols)}개: {missing_cols}')
    print('  → 해당 컬럼은 0으로 채웁니다.')
    for col in missing_cols:
        df_ext[col] = 0
else:
    print('모든 피처 컬럼 존재 ✓')

if extra_cols:
    print(f'참고: 추가 컬럼 {len(extra_cols)}개 (무시됨): {extra_cols}')

In [ ]:
# 피처 추출 (학습 시와 동일한 컬럼 순서)
X_ext = df_ext[FEATURE_COLS].copy()
y_ext = df_ext[TARGET_COL].values if HAS_LABEL else None

print(f'X shape: {X_ext.shape}')
print(f'Missing values per column:')
missing = X_ext.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '  None')
print(f'\nTotal missing: {X_ext.isnull().sum().sum()}')

## 5. Preprocessing

학습 파이프라인과 동일한 전처리:
1. Outlier clipping (continuous 컬럼만, 0.1-99.9 percentile → NaN)
2. Median imputation

In [ ]:
# Outlier detection: continuous 컬럼에 대해 0.1-99.9 percentile 기준
# NOTE: 이상적으로는 학습 데이터의 percentile bounds를 저장해서 사용해야 하지만,
#       저장되지 않았으므로 외부 데이터 자체의 percentile을 사용합니다.
#       학습 데이터의 bounds가 있다면 아래 bounds dict를 대체하세요.

continuous_cols = [col for col in FEATURE_COLS if col not in BINARY_COLS]
print(f'Continuous cols: {len(continuous_cols)}, Binary cols: {len(BINARY_COLS)}')

percentile_low = 0.001
percentile_high = 0.999

bounds = {}
for col in continuous_cols:
    q_low = X_ext[col].quantile(percentile_low)
    q_high = X_ext[col].quantile(percentile_high)
    bounds[col] = (q_low, q_high)

for col, (q_low, q_high) in bounds.items():
    outliers = ((X_ext[col] < q_low) | (X_ext[col] > q_high)).sum()
    X_ext.loc[(X_ext[col] < q_low) | (X_ext[col] > q_high), col] = np.nan

print(f'Outlier clipping 완료 (percentile {percentile_low*100:.1f}~{percentile_high*100:.1f})')
print(f'Total NaN after clipping: {X_ext.isnull().sum().sum()}')

In [ ]:
# Median imputation
imputer = SimpleImputer(strategy='median')
X_ext_np = imputer.fit_transform(X_ext)

print(f'Imputation 완료')
print(f'X shape: {X_ext_np.shape}, NaN: {np.isnan(X_ext_np).sum()}')

## 6. Load Models & Predict

In [ ]:
predictions = {}  # model_name -> y_prob array

# ---- 1) Logistic Regression ----
lr_model = joblib.load(f'{MODEL_DIR}/lr_model.joblib')
lr_scaler = joblib.load(f'{MODEL_DIR}/lr_scaler.joblib')
X_lr = lr_scaler.transform(X_ext_np)
predictions['LR'] = lr_model.predict_proba(X_lr)[:, 1]
print(f'LR  loaded & predicted ({len(predictions["LR"]):,} samples)')

# ---- 2) Random Forest ----
rf_model = joblib.load(f'{MODEL_DIR}/rf_model.joblib')
predictions['RF'] = rf_model.predict_proba(X_ext_np)[:, 1]
print(f'RF  loaded & predicted ({len(predictions["RF"]):,} samples)')

# ---- 3) XGBoost ----
xgb_model = joblib.load(f'{MODEL_DIR}/xgb_model.joblib')
predictions['XGB'] = xgb_model.predict_proba(X_ext_np)[:, 1]
print(f'XGB loaded & predicted ({len(predictions["XGB"]):,} samples)')

# ---- 4) MLP ----
mlp_scaler = joblib.load(f'{MODEL_DIR}/mlp_scaler.joblib')
checkpoint = torch.load(f'{MODEL_DIR}/mlp_model.pt', map_location='cpu', weights_only=False)

mlp_model = MLP(
    in_dim=checkpoint['in_dim'],
    hidden_dim=checkpoint['params']['hidden_dim'],
    dropout=checkpoint['params']['dropout'],
)
mlp_model.load_state_dict(checkpoint['model_state_dict'])
mlp_model.eval()

X_mlp = mlp_scaler.transform(X_ext_np)
X_mlp_t = torch.tensor(X_mlp, dtype=torch.float32)
with torch.no_grad():
    predictions['MLP'] = torch.sigmoid(mlp_model(X_mlp_t)).numpy().ravel()
print(f'MLP loaded & predicted ({len(predictions["MLP"]):,} samples)')

print(f'\nAll models loaded successfully.')

## 7. Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
colors = {'LR': '#1f77b4', 'RF': '#d62728', 'XGB': '#ff7f0e', 'MLP': '#2ca02c'}

for ax, (name, probs) in zip(axes, predictions.items()):
    ax.hist(probs, bins=50, color=colors[name], alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(THRESHOLD, color='black', linestyle='--', label=f'threshold={THRESHOLD}')
    ax.set_title(name)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')
    pos_rate = (probs >= THRESHOLD).mean() * 100
    ax.legend([f'thr={THRESHOLD} ({pos_rate:.1f}% pos)'], fontsize=9)

fig.suptitle('Prediction Probability Distribution (External Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Evaluation (라벨이 있는 경우)

In [ ]:
if not HAS_LABEL:
    print('타겟 라벨이 없으므로 평가를 건너뜁니다.')
    print('예측 결과는 predictions dict 또는 Section 10의 CSV 저장을 확인하세요.')
else:
    ext_metrics = {}
    for name, probs in predictions.items():
        ext_metrics[name] = compute_metrics(y_ext, probs, threshold=THRESHOLD)

    # Summary table
    print(f'{"Model":<6} {"AUC-ROC":>8} {"AUPRC":>8} {"PPV":>8} {"NPV":>8} {"MCC":>8} {"Sens":>8} {"Spec":>8} {"Sp@90":>8}')
    print('-' * 78)
    for name, m in ext_metrics.items():
        print(f'{name:<6} {m["auc"]:>8.4f} {m["auc_prc"]:>8.4f} {m["ppv"]:>8.4f} {m["npv"]:>8.4f} {m["mcc"]:>8.4f} {m["sensitivity"]:>8.4f} {m["specificity"]:>8.4f} {m["spec@90"]:>8.4f}')

## 9. Visualization (라벨이 있는 경우)

In [ ]:
if not HAS_LABEL:
    print('타겟 라벨이 없으므로 시각화를 건너뜁니다.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ---- ROC Curves ----
    ax = axes[0]
    for name, m in ext_metrics.items():
        ax.plot(m['fpr'], m['tpr'], color=colors[name],
                label=f"{name} (AUC={m['auc']:.4f})", linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — External Validation')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    # ---- PR Curves ----
    ax = axes[1]
    baseline = y_ext.mean()
    for name, m in ext_metrics.items():
        ax.plot(m['recall_curve'], m['precision_curve'], color=colors[name],
                label=f"{name} (AUPRC={m['auc_prc']:.4f})", linewidth=2)
    ax.axhline(baseline, color='black', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.4f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curves — External Validation')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/external_validation_curves.png', dpi=150, bbox_inches='tight')
    print(f'Figure saved to {MODEL_DIR}/external_validation_curves.png')
    plt.show()

## 9-1. Internal vs External 비교 (라벨이 있는 경우)

In [ ]:
if not HAS_LABEL:
    print('타겟 라벨이 없으므로 비교를 건너뜁니다.')
else:
    # Load internal test metrics
    internal_metrics = {}
    for name in ['LR', 'RF', 'XGB', 'MLP']:
        fpath = f'{MODEL_DIR}/{name}_test_metrics.json'
        if os.path.exists(fpath):
            with open(fpath) as f:
                internal_metrics[name] = json.load(f)

    if internal_metrics:
        print(f'{"":<6} {"--- Internal (MIMIC-IV Test) ---":>36}   {"--- External Validation ---":>32}')
        print(f'{"Model":<6} {"AUC-ROC":>8} {"AUPRC":>8} {"MCC":>8} {"Sens":>8}   {"AUC-ROC":>8} {"AUPRC":>8} {"MCC":>8} {"Sens":>8}   {"ΔAUC":>7} {"ΔAUPRC":>7}')
        print('-' * 110)
        for name in ['LR', 'RF', 'XGB', 'MLP']:
            if name in internal_metrics and name in ext_metrics:
                im = internal_metrics[name]
                em = ext_metrics[name]
                d_auc = em['auc'] - im['auc']
                d_auprc = em['auc_prc'] - im['auc_prc']
                print(f'{name:<6} {im["auc"]:>8.4f} {im["auc_prc"]:>8.4f} {im["mcc"]:>8.4f} {im["sensitivity"]:>8.4f}   '
                      f'{em["auc"]:>8.4f} {em["auc_prc"]:>8.4f} {em["mcc"]:>8.4f} {em["sensitivity"]:>8.4f}   '
                      f'{d_auc:>+7.4f} {d_auprc:>+7.4f}')
    else:
        print('Internal test metrics not found. Skipping comparison.')

In [ ]:
if HAS_LABEL and internal_metrics:
    metric_names = ['AUC-ROC', 'AUPRC', 'PPV', 'MCC', 'Sensitivity', 'Spec@90']
    metric_keys = ['auc', 'auc_prc', 'ppv', 'mcc', 'sensitivity', 'spec@90']
    model_names = ['LR', 'RF', 'XGB', 'MLP']

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    x = np.arange(len(model_names))
    width = 0.35

    for idx, (mname, mkey) in enumerate(zip(metric_names, metric_keys)):
        ax = axes[idx]
        int_vals = [internal_metrics.get(m, {}).get(mkey, 0) for m in model_names]
        ext_vals = [ext_metrics.get(m, {}).get(mkey, 0) for m in model_names]

        bars1 = ax.bar(x - width/2, int_vals, width, label='Internal', color='#4C72B0', alpha=0.8)
        bars2 = ax.bar(x + width/2, ext_vals, width, label='External', color='#DD8452', alpha=0.8)

        ax.set_ylabel(mname)
        ax.set_title(mname)
        ax.set_xticks(x)
        ax.set_xticklabels(model_names)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        for bar in bars1:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

    fig.suptitle('Internal vs External Validation', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/internal_vs_external_comparison.png', dpi=150, bbox_inches='tight')
    print(f'Figure saved to {MODEL_DIR}/internal_vs_external_comparison.png')
    plt.show()

## 10. Save Results

In [ ]:
# 예측 확률을 CSV로 저장
result_df = df_ext[['stay_id']].copy() if 'stay_id' in df_ext.columns else pd.DataFrame(index=df_ext.index)
if HAS_LABEL:
    result_df[TARGET_COL] = y_ext

for name, probs in predictions.items():
    result_df[f'{name}_prob'] = probs
    result_df[f'{name}_pred'] = (probs >= THRESHOLD).astype(int)

output_path = f'{MODEL_DIR}/external_validation_predictions.csv'
result_df.to_csv(output_path, index=False)
print(f'Predictions saved to {output_path}')
print(f'Shape: {result_df.shape}')
result_df.head()

In [ ]:
# 메트릭을 JSON으로 저장 (라벨 있는 경우)
if HAS_LABEL:
    ext_results = {}
    for name, m in ext_metrics.items():
        ext_results[name] = {
            k: float(v) for k, v in m.items()
            if isinstance(v, (int, float, np.floating))
        }

    metrics_path = f'{MODEL_DIR}/external_validation_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(ext_results, f, indent=2)
    print(f'Metrics saved to {metrics_path}')
    print(json.dumps(ext_results, indent=2))
else:
    print('No labels — metrics not saved.')

## 11. Calibration Plot (라벨이 있는 경우)

In [ ]:
if not HAS_LABEL:
    print('타겟 라벨이 없으므로 calibration plot을 건너뜁니다.')
else:
    from sklearn.calibration import calibration_curve

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')

    for name, probs in predictions.items():
        fraction_pos, mean_predicted = calibration_curve(y_ext, probs, n_bins=10, strategy='uniform')
        ax.plot(mean_predicted, fraction_pos, 's-', color=colors[name], label=name, linewidth=2)

    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title('Calibration Plot — External Validation')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/external_calibration.png', dpi=150, bbox_inches='tight')
    print(f'Figure saved to {MODEL_DIR}/external_calibration.png')
    plt.show()